Evaluation: figures and tables of the report. Run every cell in order (Run All). The notebook only reads results/ and writes figures to notebooks/figures and tables to notebooks/tables.

Imports.

In [1]:
import json
import re
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from scipy import stats

Folders. The notebook can be started from the top folder or from notebooks/.

In [2]:
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from sflad.data.event_io import events_from_jsonl
from sflad.detectors.base import lexical_window_features
from sflad.pipeline.analysis import holm as holm_family

RUNS = ROOT / "results" / "runs"
AN = ROOT / "results" / "analysis"
FIG = ROOT / "notebooks" / "figures"
TAB = ROOT / "notebooks" / "tables"
FIG.mkdir(parents=True, exist_ok=True)
TAB.mkdir(parents=True, exist_ok=True)

Settings read from configs/default.yaml.

In [3]:
CFG = yaml.safe_load((ROOT / "configs" / "default.yaml").read_text())
WINDOW = CFG["features"]["window_size"]
TARGET = CFG["detectors"]["calib_fpr"]
MARGIN = CFG["analysis"]["viable_forfeiture_f1"]
print("window size:", WINDOW, "lines; false-alarm target:", TARGET, "; margin:", MARGIN)

window size: 20 lines; false-alarm target: 0.01 ; margin: 0.1


Runs, applications, detectors and fault categories.

In [4]:
APPS = ["orders", "bookings"]
RUN_TAGS = [f"run{i}" for i in range(1, 10)]
SET_OF = {}
for i, r in enumerate(RUN_TAGS):
    SET_OF[r] = i // 3 + 1

ARMS = ["source_free", "source_free_deeplog", "transfer", "transfer_openstack", "transfer_similar", "threshold"]
ABLATIONS = ["source_free_forest", "novelty_gate", "source_free_source_features"]
LABEL = {
    "source_free": "Isolation Forest (source-free)",
    "source_free_deeplog": "DeepLog-style (source-free)",
    "transfer": "BGL-trained baseline",
    "transfer_openstack": "OpenStack-trained baseline",
    "transfer_similar": "Similar-application baseline",
    "threshold": "Threshold alarms",
    "source_free_forest": "Isolation Forest alone",
    "novelty_gate": "Novelty gate alone",
    "source_free_source_features": "Source-free detector on the BGL baseline's features",
}
SF = {"source_free": "Isolation Forest", "source_free_deeplog": "DeepLog-style"}
BASE = {"H1": "transfer", "H2": "transfer_openstack", "H3": "transfer_similar"}
CATS = ["permission", "config", "timeout", "resource", "latency"]

Short detector names for the figure axes, and the colours.

In [5]:
SHORT = ["Isolation\nForest", "DeepLog-\nstyle", "BGL\nbaseline", "OpenStack\nbaseline", "Similar-app\nbaseline", "Threshold\nalarms"]

def flat(i):
    return SHORT[i].replace("-\n", "-").replace("\n", " ")

DET_COLOR = dict(zip(ARMS + ABLATIONS, ["tab:blue", "tab:orange", "tab:green", "tab:brown", "tab:red", "tab:purple",
                                        "tab:cyan", "tab:olive", "tab:gray"]))
SET_COLOR = {1: "#1b9e77", 2: "#d95f02", 3: "#7570b3"}
plt.rcParams.update({"font.size": 9, "figure.dpi": 120, "savefig.bbox": "tight", "axes.titlesize": 9,
                     "figure.titlesize": 10, "axes.spines.top": False, "axes.spines.right": False})

Helper functions.

In [6]:
def load(path):
    return json.loads(Path(path).read_text())

def ratio(a, b):
    # a / b, or NaN when b is 0 (an undefined rate, never 0)
    if b:
        return a / b
    return float("nan")

def save(df, name):
    # N/A marks no value: a rate over zero cases, a test with no disagreement, or a summary row
    df.to_csv(TAB / f"{name}.csv", index=False, na_rep="N/A")
    return df

def suptitle(fig, text):
    fig.suptitle(text, y=1.04)

Load the pooled analyses and every run's scores and window predictions.

In [7]:
analysis = {}
for app in APPS:
    analysis[app] = load(AN / "pooled" / app / "analysis.json")

results = {}
preds = {}
for r in RUN_TAGS:
    for app in APPS:
        results[(r, app)] = load(RUNS / r / app / "processed" / "results.json")
        preds[(r, app)] = load(RUNS / r / app / "processed" / "window_predictions.json")
print(len(results), "run and application pairs loaded")

18 run and application pairs loaded


1. Execution counts per run and application (table execution_counts).

In [8]:
rows = []
for r in RUN_TAGS:
    for app in APPS:
        raw = RUNS / r / app / "raw"
        ex = load(raw / "export.json")
        ev = events_from_jsonl(raw / "events.jsonl")
        train = sum(e.split == "train" for e in ev)
        held = sum(e.split == "eval" for e in ev)
        w = preds[(r, app)]
        n_win = len(w["y_true"])
        eps = ex["episodes"]
        reasons = ex["fault_reasons"]
        rows.append({
            "run": r,
            "application": app,
            "invocations": ex["invocations"],
            "log_lines": ex["lines"],
            "training_lines": train,
            "scored_lines": n_win * WINDOW,
            "unscored_lines": held - n_win * WINDOW,
            "windows": n_win,
            "fault_windows": int(sum(w["y_true"])),
            "elasticity_windows": int(sum(w["control_windows"])),
            "fault_episodes": sum(v["episodes"] for v in eps.values()),
            "slow_limit_ms": ex["slow_limit_ms"],
            "faulty_calls_in_faults": ex["faulty_invocations"].get("fault", 0),
            "faulty_calls_outside_faults": sum(v for k, v in ex["faulty_invocations"].items() if k != "fault"),
            "slow_calls_outside_faults": sum(v for k, v in reasons.items() if k.endswith(":slow") and not k.startswith("fault:")),
            "incorrect_answers": sum(v for k, v in reasons.items() if k.endswith(":incorrect")),
            "silent_episodes": sum(v["silent"] for v in eps.values()),
        })
counts = save(pd.DataFrame(rows), "execution_counts")
counts

,run,application,invocations,log_lines,training_lines,scored_lines,unscored_lines,windows,fault_windows,elasticity_windows,fault_episodes,slow_limit_ms,faulty_calls_in_faults,faulty_calls_outside_faults,slow_calls_outside_faults,incorrect_answers,silent_episodes
0,run1,orders,7106,36059,6221,29820,18,1491,444,90,300,3721,600,1,1,0,0
1,run1,bookings,7120,36116,6236,29880,0,1494,448,90,300,3116,598,0,0,0,1
2,run2,orders,7127,36154,6205,29940,9,1497,448,91,300,2746,599,0,0,0,0
3,run2,bookings,7122,36132,6245,29880,7,1494,439,90,300,3329,600,0,0,0,0
4,run3,orders,7122,36128,6210,29900,18,1495,442,90,300,2727,600,0,0,0,0
5,run3,bookings,7114,36097,6231,29860,6,1493,442,91,300,3054,600,0,0,0,0
6,run4,orders,7099,36019,6208,29800,11,1490,448,91,300,2694,600,0,0,0,0
7,run4,bookings,7098,36006,6135,29860,11,1493,445,90,300,2913,598,0,0,0,1
8,run5,orders,7117,36104,6215,29880,9,1494,433,91,300,2686,600,0,0,0,0
9,run5,bookings,7114,36085,6245,29840,0,1492,444,92,300,2896,600,0,0,0,0


Check: every log line is a training, scored or unscored line, and fewer than one window of lines is left unscored.

In [9]:
assert (counts.training_lines + counts.scored_lines + counts.unscored_lines == counts.log_lines).all()
assert counts.unscored_lines.between(0, WINDOW - 1).all()
print("line counts check passed")

line counts check passed


2. Pooled precision, recall, F1 and MCC per detector over the nine runs (table pooled_metrics).

In [10]:
rows = []
for app in APPS:
    for arm in ARMS:
        p = analysis[app]["pooled"][arm]
        rows.append({"application": app, "detector": LABEL[arm], "precision": p["precision"], "recall": p["recall"],
                     "f1": p["f1"], "mcc": p["mcc"], "windows": p["n"], "fault_windows": p["support_pos"]})
pooled = save(pd.DataFrame(rows), "pooled_metrics")
pooled.round(4)

,application,detector,precision,recall,f1,mcc,windows,fault_windows
0,orders,Isolation Forest (source-free),0.9606,0.7440,0.8386,0.7942,13438,3969
1,orders,DeepLog-style (source-free),0.8543,0.8818,0.8678,0.8114,13438,3969
2,orders,BGL-trained baseline,0.9989,0.6982,0.8219,0.7866,13438,3969
3,orders,OpenStack-trained baseline,1.0000,0.3406,0.5082,0.5166,13438,3969
4,orders,Similar-application baseline,0.9990,0.7344,0.8465,0.8123,13438,3969
5,orders,Threshold alarms,1.0000,0.2063,0.3421,0.3935,13438,3969
6,bookings,Isolation Forest (source-free),0.9721,0.7201,0.8273,0.7851,13442,3965
7,bookings,DeepLog-style (source-free),0.8884,0.7753,0.8280,0.7662,13442,3965
8,bookings,BGL-trained baseline,0.9993,0.6837,0.8119,0.7766,13442,3965
9,bookings,OpenStack-trained baseline,1.0000,0.3342,0.5009,0.5112,13442,3965


Detection delay per detector over the fault segments (table detection_delay).

In [11]:
rows = []
for app in APPS:
    for arm in ARMS:
        d = analysis[app]["detection_delay"][arm]
        rows.append({"application": app, "detector": LABEL[arm], "segments": d["n_segments"],
                     "segments_detected": d["n_detected_segments"], "mean_delay_windows": d["mean_delay_windows"],
                     "median_delay_s": d["median_delay_seconds"], "mean_delay_s": d["mean_delay_seconds"]})
delay = save(pd.DataFrame(rows), "detection_delay")
delay.round(2)

,application,detector,segments,segments_detected,mean_delay_windows,median_delay_s,mean_delay_s
0,orders,Isolation Forest (source-free),2701,2330,0.14,0.0,1.69
1,orders,DeepLog-style (source-free),2701,2700,0.17,0.0,2.09
2,orders,BGL-trained baseline,2701,2160,0.13,0.0,1.44
3,orders,OpenStack-trained baseline,2701,1087,0.15,0.0,0.99
4,orders,Similar-application baseline,2701,2304,0.13,0.0,1.49
5,orders,Threshold alarms,2701,819,0.20,0.0,1.31
6,bookings,Isolation Forest (source-free),2697,2278,0.15,0.0,1.94
7,bookings,DeepLog-style (source-free),2697,2405,0.17,0.0,2.21
8,bookings,BGL-trained baseline,2697,2158,0.15,0.0,1.91
9,bookings,OpenStack-trained baseline,2697,1088,0.17,0.0,1.20


Figure pooled_metrics.pdf: the four measures per detector.

In [12]:
METRICS = [("precision", "Precision"), ("recall", "Recall"), ("f1", "F1"), ("mcc", "MCC")]
M_COLOR = ["#9ecae1", "#4292c6", "#08519c", "#636363"]
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.2), sharey=True)
for ax, app in zip(axes, APPS):
    d = pooled[pooled.application == app].set_index("detector")
    x = np.arange(len(ARMS))
    for j, (m, name) in enumerate(METRICS):
        vals = [d.loc[LABEL[a], m] for a in ARMS]
        bars = ax.bar(x + (j - 1.5) * 0.2, vals, 0.2, color=M_COLOR[j], label=name)
        for b in bars:
            ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.01, f"{b.get_height():.2f}",
                    ha="center", va="bottom", fontsize=5.5, rotation=90)
    ax.set_xticks(x)
    ax.set_xticklabels(SHORT, fontsize=7)
    ax.set_title(app.capitalize())
    ax.set_ylim(min(0, pooled[[m for m, _ in METRICS]].min().min() - 0.05), 1.15)
axes[0].set_ylabel("Pooled value (0 to 1)")
h, l = axes[0].get_legend_handles_labels()
fig.legend(h, l, loc="lower center", ncol=4, fontsize=7, bbox_to_anchor=(0.5, -0.08))
suptitle(fig, "Pooled precision, recall, F1 and MCC per detector (nine runs)")
fig.savefig(FIG / "pooled_metrics.pdf")
plt.close(fig)

Summary of the pooled measures.

In [13]:
for app in APPS:
    d = pooled[pooled.application == app].set_index("detector")
    learned = [LABEL[a] for a in ARMS if a != "threshold"]
    th = LABEL["threshold"]
    dl = LABEL["source_free_deeplog"]
    print(f"{app}: F1 of the four learned detectors {d.loc[learned, 'f1'].min():.3f} to {d.loc[learned, 'f1'].max():.3f}; "
          f"threshold alarms F1 {d.loc[th, 'f1']:.3f} (precision {d.loc[th, 'precision']:.3f}, recall {d.loc[th, 'recall']:.3f}); "
          f"DeepLog-style precision {d.loc[dl, 'precision']:.3f} vs recall {d.loc[dl, 'recall']:.3f}")

orders: F1 of the four learned detectors 0.508 to 0.868; threshold alarms F1 0.342 (precision 1.000, recall 0.206); DeepLog-style precision 0.854 vs recall 0.882
bookings: F1 of the four learned detectors 0.501 to 0.828; threshold alarms F1 0.353 (precision 1.000, recall 0.214); DeepLog-style precision 0.888 vs recall 0.775


Pooled confusion counts per detector (table confusion_counts).

In [14]:
rows = []
for app in APPS:
    for arm in ARMS:
        p = analysis[app]["pooled"][arm]
        rows.append({"application": app, "detector": LABEL[arm], "tp": p["tp"], "fn": p["fn"], "fp": p["fp"], "tn": p["tn"],
                     "fault_windows": p["support_pos"], "fault_free_windows": p["n"] - p["support_pos"]})
conf = save(pd.DataFrame(rows), "confusion_counts")
assert (conf.tp + conf.fn == conf.fault_windows).all()
assert (conf.fp + conf.tn == conf.fault_free_windows).all()
conf

,application,detector,tp,fn,fp,tn,fault_windows,fault_free_windows
0,orders,Isolation Forest (source-free),2953,1016,121,9348,3969,9469
1,orders,DeepLog-style (source-free),3500,469,597,8872,3969,9469
2,orders,BGL-trained baseline,2771,1198,3,9466,3969,9469
3,orders,OpenStack-trained baseline,1352,2617,0,9469,3969,9469
4,orders,Similar-application baseline,2915,1054,3,9466,3969,9469
5,orders,Threshold alarms,819,3150,0,9469,3969,9469
6,bookings,Isolation Forest (source-free),2855,1110,82,9395,3965,9477
7,bookings,DeepLog-style (source-free),3074,891,386,9091,3965,9477
8,bookings,BGL-trained baseline,2711,1254,2,9475,3965,9477
9,bookings,OpenStack-trained baseline,1325,2640,0,9477,3965,9477


Figure confusion_counts.pdf: cell colour is the share of the true class.

In [15]:
fig, axes = plt.subplots(1, 2, figsize=(8, 2.9), sharey=True)
for ax, app in zip(axes, APPS):
    d = conf[conf.application == app].set_index("detector").loc[[LABEL[a] for a in ARMS]]
    cnt = d[["tp", "fn", "fp", "tn"]].to_numpy()
    share = np.column_stack([cnt[:, 0] / d.fault_windows, cnt[:, 1] / d.fault_windows,
                             cnt[:, 2] / d.fault_free_windows, cnt[:, 3] / d.fault_free_windows])
    im = ax.imshow(share, cmap="Blues", vmin=0, vmax=1, aspect="auto")
    for i in range(cnt.shape[0]):
        for j in range(4):
            color = "white" if share[i, j] > 0.6 else "black"
            ax.text(j, i, f"{cnt[i, j]:,}", ha="center", va="center", fontsize=7, color=color)
    ax.set_xticks(range(4))
    ax.set_xticklabels(["TP", "FN", "FP", "TN"])
    ax.set_title(app.capitalize())
    ax.set_yticks(range(len(ARMS)))
    ax.set_yticklabels([flat(i) for i in range(len(ARMS))], fontsize=7)
    ax.tick_params(length=0)
    for s in ax.spines.values():
        s.set_visible(False)
cb = fig.colorbar(im, ax=axes, fraction=0.03, pad=0.02)
cb.set_label("Share of the true class", fontsize=7)
cb.ax.tick_params(labelsize=7)
suptitle(fig, "Pooled confusion counts per detector (windows, nine runs)")
fig.savefig(FIG / "confusion_counts.pdf")
plt.close(fig)
for app in APPS:
    d = conf[conf.application == app].set_index("detector")
    parts = [f"{flat(i)} FP {d.loc[LABEL[a], 'fp']}, FN {d.loc[LABEL[a], 'fn']}" for i, a in enumerate(ARMS)]
    print(f"{app}: " + "; ".join(parts))

orders: Isolation Forest FP 121, FN 1016; DeepLog-style FP 597, FN 469; BGL baseline FP 3, FN 1198; OpenStack baseline FP 0, FN 2617; Similar-app baseline FP 3, FN 1054; Threshold alarms FP 0, FN 3150
bookings: Isolation Forest FP 82, FN 1110; DeepLog-style FP 386, FN 891; BGL baseline FP 2, FN 1254; OpenStack baseline FP 0, FN 2640; Similar-app baseline FP 1, FN 1204; Threshold alarms FP 0, FN 3116


3. Pooled forfeiture (baseline F1 minus source-free F1) with its 95% interval, non-inferiority and viability (table forfeiture_pooled).

In [16]:
rows = []
for app in APPS:
    for fam in SF:
        for h, base in BASE.items():
            t = analysis[app]["families"][fam]["tests"][h]
            d = analysis[app]["families"][fam]["decisions"][base]
            rows.append({"application": app, "source_free": SF[fam], "baseline": LABEL[base], "forfeiture": t["gap"],
                         "ci_low": t["ci_low"], "ci_high": t["ci_high"], "upper_one_sided_95": t["upper_one_sided_95"],
                         "non_inferior": d["non_inferior"], "viable": d["viable_substitute"]})
forf = save(pd.DataFrame(rows), "forfeiture_pooled")
forf.round(4)

,application,source_free,baseline,forfeiture,ci_low,ci_high,upper_one_sided_95,non_inferior,viable
0,orders,Isolation Forest,BGL-trained baseline,-0.0167,-0.0217,-0.0115,-0.0123,True,True
1,orders,Isolation Forest,OpenStack-trained baseline,-0.3304,-0.3455,-0.3150,-0.3172,True,True
2,orders,Isolation Forest,Similar-application baseline,0.0080,0.0021,0.0143,0.0131,True,True
3,orders,DeepLog-style,BGL-trained baseline,-0.0460,-0.0560,-0.0356,-0.0371,True,True
4,orders,DeepLog-style,OpenStack-trained baseline,-0.3597,-0.3755,-0.3423,-0.3456,True,True
5,orders,DeepLog-style,Similar-application baseline,-0.0213,-0.0309,-0.0118,-0.0133,True,True
6,bookings,Isolation Forest,BGL-trained baseline,-0.0154,-0.0200,-0.0109,-0.0116,True,True
7,bookings,Isolation Forest,OpenStack-trained baseline,-0.3264,-0.3409,-0.3117,-0.3137,True,True
8,bookings,Isolation Forest,Similar-application baseline,-0.0064,-0.0116,-0.0013,-0.0023,True,True
9,bookings,DeepLog-style,BGL-trained baseline,-0.0161,-0.0234,-0.0086,-0.0096,True,True


Figure forfeiture_pooled.pdf (Figure 2 of the report).

In [17]:
pairs = [(s, LABEL[b]) for s in ["Isolation Forest", "DeepLog-style"] for b in BASE.values()]
fig, ax = plt.subplots(figsize=(4.8, 2.3))
for k, (app, mk, off) in enumerate((("orders", "o", 0.12), ("bookings", "s", -0.12))):
    d = forf[forf.application == app].set_index(["source_free", "baseline"])
    y = np.arange(len(pairs))[::-1] + off
    v = np.array([d.loc[pp, "forfeiture"] for pp in pairs])
    lo = np.array([d.loc[pp, "ci_low"] for pp in pairs])
    hi = np.array([d.loc[pp, "ci_high"] for pp in pairs])
    color = "black" if k == 0 else "tab:blue"
    ax.errorbar(v, y, xerr=[v - lo, hi - v], fmt=mk, ms=4, capsize=3, color=color, label=app.capitalize())
ax.axvline(0, color="grey", lw=0.8)
ax.axvline(MARGIN, color="grey", ls="--", lw=0.8)
ax.text(MARGIN + 0.005, 1.5, "non-inferiority\n" + f"margin {MARGIN:.2f}", fontsize=8, color="grey", va="center", ha="left")
ax.set_yticks(np.arange(len(pairs))[::-1])
ax.set_yticklabels([f"{s} vs\n{b}" for s, b in pairs], fontsize=8)
ax.set_xlim(min(0, forf.ci_low.min()) - 0.005, max(MARGIN, forf.ci_high.max()) + 0.085)
ax.set_xlabel("F1 forfeiture (baseline F1 minus source-free F1), 95% interval", fontsize=8)
ax.legend(fontsize=8, loc="upper right")
fig.savefig(FIG / "forfeiture_pooled.pdf")
plt.close(fig)

Summary of the pooled forfeitures.

In [18]:
print(f"largest upper interval bound: {forf.ci_high.max():.4f}; smallest lower bound: {forf.ci_low.min():.4f}; "
      f"viable pairs: {int(forf.viable.sum())} of {len(forf)}")
for fam in SF.values():
    d = forf[forf.source_free == fam]
    print(f"{fam}: forfeiture {d.forfeiture.min():.4f} to {d.forfeiture.max():.4f}")

largest upper interval bound: 0.0143; smallest lower bound: -0.3755; viable pairs: 12 of 12
Isolation Forest: forfeiture -0.3304 to 0.0080
DeepLog-style: forfeiture -0.3597 to -0.0071


4. F1 and MCC of every detector in every run (table per_run_f1).

In [19]:
rows = []
for app in APPS:
    for r in RUN_TAGS:
        s = results[(r, app)]["summaries"]
        for arm in ARMS:
            rows.append({"application": app, "run": r, "detector": LABEL[arm], "f1": s[arm]["f1"], "mcc": s[arm]["mcc"]})
per_run = save(pd.DataFrame(rows), "per_run_f1")
per_run.round(4)

,application,run,detector,f1,mcc
0,orders,run1,Isolation Forest (source-free),0.8290,0.7757
1,orders,run1,DeepLog-style (source-free),0.8674,0.8110
2,orders,run1,BGL-trained baseline,0.8181,0.7815
3,orders,run1,OpenStack-trained baseline,0.5025,0.5117
4,orders,run1,Similar-application baseline,0.8819,0.8502
...,...,...,...,...,...
103,bookings,run9,DeepLog-style (source-free),0.8514,0.7884
104,bookings,run9,BGL-trained baseline,0.8189,0.7845
105,bookings,run9,OpenStack-trained baseline,0.5111,0.5196
106,bookings,run9,Similar-application baseline,0.8347,0.8007


Figure per_run_f1.pdf: one dot per run, coloured by its set of three runs.

In [20]:
fig, axes = plt.subplots(1, 2, figsize=(8, 3.2), sharey=True)
for ax, app in zip(axes, APPS):
    d = per_run[per_run.application == app]
    for i, arm in enumerate(ARMS):
        dd = d[d.detector == LABEL[arm]]
        xs = np.full(len(dd), i) + np.linspace(-0.2, 0.2, len(dd))
        ax.scatter(xs, dd.f1, s=12, c=[SET_COLOR[SET_OF[r]] for r in dd.run])
    ax.set_xticks(range(len(ARMS)))
    ax.set_xticklabels(SHORT, fontsize=7)
    ax.set_title(app.capitalize())
axes[0].set_ylabel("F1 per run")
handles = [Line2D([], [], marker="o", ls="", color=SET_COLOR[s], label=f"Set {s} (runs {3*s-2} to {3*s})") for s in (1, 2, 3)]
fig.legend(handles=handles, loc="lower center", ncol=3, fontsize=7, bbox_to_anchor=(0.5, -0.08))
suptitle(fig, "F1 of each detector in each of the nine runs")
fig.savefig(FIG / "per_run_f1.pdf")
plt.close(fig)

Range of F1 over the runs, the lowest learned detector per run, and whether the sets overlap.

In [21]:
rng = per_run.groupby(["application", "detector"]).f1.agg(["min", "max"])
rng["range"] = rng["max"] - rng["min"]
print(rng.round(3))
learned_runs = per_run[per_run.detector != LABEL["threshold"]]
lowest = learned_runs.loc[learned_runs.groupby(["application", "run"]).f1.idxmin()].detector.value_counts()
print("lowest learned detector per run and application:", lowest.to_dict())
with_set = per_run.assign(set=per_run.run.map(SET_OF))
sr = with_set.groupby(["application", "detector", "set"]).f1.agg(["min", "max"]).groupby(level=[0, 1])
print("per-set F1 ranges overlap for every detector:", all(g["min"].max() <= g["max"].min() for _, g in sr))

                                              min    max  range
application detector                                           
bookings    BGL-trained baseline            0.789  0.825  0.036
            DeepLog-style (source-free)     0.779  0.873  0.094
            Isolation Forest (source-free)  0.815  0.853  0.038
            OpenStack-trained baseline      0.474  0.511  0.037
            Similar-application baseline    0.789  0.856  0.067
            Threshold alarms                0.335  0.365  0.029
orders      BGL-trained baseline            0.814  0.836  0.022
            DeepLog-style (source-free)     0.861  0.885  0.024
            Isolation Forest (source-free)  0.799  0.879  0.079
            OpenStack-trained baseline      0.494  0.519  0.025
            Similar-application baseline    0.812  0.883  0.071
            Threshold alarms                0.313  0.366  0.053
lowest learned detector per run and application: {'OpenStack-trained baseline': 18}
per-set F1 ranges ov

Forfeiture inside each run (table forfeiture_per_run).

In [22]:
rows = []
for app in APPS:
    for fam in SF:
        for h, base in BASE.items():
            for r in RUN_TAGS:
                s = results[(r, app)]["summaries"]
                rows.append({"application": app, "source_free": SF[fam], "baseline": LABEL[base], "run": r, "set": SET_OF[r],
                             "forfeiture": s[base]["f1"] - s[fam]["f1"]})
per_run_forf = save(pd.DataFrame(rows), "forfeiture_per_run")
per_run_forf.round(4)

,application,source_free,baseline,run,set,forfeiture
0,orders,Isolation Forest,BGL-trained baseline,run1,1,-0.0109
1,orders,Isolation Forest,BGL-trained baseline,run2,1,-0.0051
2,orders,Isolation Forest,BGL-trained baseline,run3,1,0.0008
3,orders,Isolation Forest,BGL-trained baseline,run4,2,-0.0113
4,orders,Isolation Forest,BGL-trained baseline,run5,2,-0.0005
...,...,...,...,...,...,...
103,bookings,DeepLog-style,Similar-application baseline,run5,2,0.0219
104,bookings,DeepLog-style,Similar-application baseline,run6,2,-0.0517
105,bookings,DeepLog-style,Similar-application baseline,run7,3,-0.0088
106,bookings,DeepLog-style,Similar-application baseline,run8,3,0.0151


Figure forfeiture_per_run.pdf: dots are runs, the bar is the pooled value. IF = Isolation Forest, DL = DeepLog-style.

In [23]:
PAIR_TICK = [f"{s} vs\n{b}" for s in ("IF", "DL") for b in ("BGL", "OpenStack", "similar")]
fig, axes = plt.subplots(1, 2, figsize=(7, 2.6), sharey=True)
for ax, app in zip(axes, APPS):
    for i, (s, b) in enumerate(pairs):
        d = per_run_forf[(per_run_forf.application == app) & (per_run_forf.source_free == s) & (per_run_forf.baseline == b)]
        ax.scatter(i + np.linspace(-0.2, 0.2, len(d)), d.forfeiture, s=12, c=[SET_COLOR[k] for k in d["set"]], zorder=3)
        pv = forf[(forf.application == app) & (forf.source_free == s) & (forf.baseline == b)].forfeiture.iloc[0]
        ax.hlines(pv, i - 0.3, i + 0.3, color="black", lw=1.5, zorder=4)
    ax.axhline(0, color="grey", lw=0.8)
    ax.axhline(MARGIN, color="grey", ls="--", lw=0.8)
    ax.set_xticks(range(len(pairs)))
    ax.set_xticklabels(PAIR_TICK, fontsize=7)
    ax.set_title(app.capitalize())
    ax.set_ylim(min(-0.03, per_run_forf.forfeiture.min() - 0.01), max(MARGIN + 0.03, per_run_forf.forfeiture.max() + 0.015))
axes[0].set_ylabel("F1 forfeiture in the run\n(baseline F1 minus source-free F1)")
handles = [Line2D([], [], marker="o", ls="", color=SET_COLOR[s], label=f"Set {s}") for s in (1, 2, 3)]
handles += [Line2D([], [], color="black", lw=1.5, label="Pooled (nine runs)"),
            Line2D([], [], color="grey", ls="--", lw=0.8, label=f"Viability margin {MARGIN:.2f}")]
fig.legend(handles=handles, loc="lower center", ncol=5, fontsize=6.5, bbox_to_anchor=(0.5, -0.13),
           handletextpad=0.3, columnspacing=0.8)
fig.savefig(FIG / "forfeiture_per_run.pdf")
plt.close(fig)

Summary of the forfeitures per run and per set.

In [24]:
g = per_run_forf.groupby(["application", "source_free", "baseline"]).forfeiture
summary = pd.DataFrame({"min": g.min(), "max": g.max(),
                        "runs_below_0": g.apply(lambda v: int((v < 0).sum())),
                        "runs_above_margin": g.apply(lambda v: int((v > MARGIN).sum()))})
print(summary.round(4))
set_means = per_run_forf.groupby(["application", "source_free", "baseline", "set"]).forfeiture.mean().unstack()
print(set_means.round(4).rename(columns=lambda k: f"set {k} mean"))
above = per_run_forf[per_run_forf.forfeiture > MARGIN]
print("runs above the margin:")
print(above[["application", "source_free", "baseline", "run", "forfeiture"]].round(4).to_string(index=False))

                                                              min     max  \
application source_free      baseline                                       
bookings    DeepLog-style    BGL-trained baseline         -0.0614  0.0256   
                             OpenStack-trained baseline   -0.3754 -0.2812   
                             Similar-application baseline -0.0517  0.0292   
            Isolation Forest BGL-trained baseline         -0.0363  0.0001   
                             OpenStack-trained baseline   -0.3519 -0.3081   
                             Similar-application baseline -0.0363  0.0294   
orders      DeepLog-style    BGL-trained baseline         -0.0664 -0.0251   
                             OpenStack-trained baseline   -0.3688 -0.3488   
                             Similar-application baseline -0.0622  0.0213   
            Isolation Forest BGL-trained baseline         -0.0643  0.0151   
                             OpenStack-trained baseline   -0.3703 -0.2893   

Baseline MCC minus source-free MCC per run, checked against each run's results.json (table mcc_difference_per_run).

In [25]:
rows = []
for app in APPS:
    for fam in SF:
        for base in BASE.values():
            d = [results[(r, app)]["summaries"][base]["mcc"] - results[(r, app)]["summaries"][fam]["mcc"] for r in RUN_TAGS]
            recorded = [results[(r, app)]["forfeiture"][f"mcc:{base}-{fam}"] for r in RUN_TAGS]
            assert np.allclose(d, recorded)
            rows.append({"application": app, "source_free": SF[fam], "baseline": LABEL[base],
                         "mean": np.mean(d), "min": np.min(d), "max": np.max(d)})
mcc_diff = save(pd.DataFrame(rows), "mcc_difference_per_run")
mcc_diff.round(4)

,application,source_free,baseline,mean,min,max
0,orders,Isolation Forest,BGL-trained baseline,-0.0079,-0.0603,0.0278
1,orders,Isolation Forest,OpenStack-trained baseline,-0.2780,-0.3223,-0.2334
2,orders,Isolation Forest,Similar-application baseline,0.0178,-0.0643,0.0745
3,orders,DeepLog-style,BGL-trained baseline,-0.0252,-0.0560,-0.0000
4,orders,DeepLog-style,OpenStack-trained baseline,-0.2952,-0.3138,-0.2826
5,orders,DeepLog-style,Similar-application baseline,0.0005,-0.0508,0.0502
6,bookings,Isolation Forest,BGL-trained baseline,-0.0086,-0.0261,0.0062
7,bookings,Isolation Forest,OpenStack-trained baseline,-0.2740,-0.3003,-0.2614
8,bookings,Isolation Forest,Similar-application baseline,0.0006,-0.0261,0.0333
9,bookings,DeepLog-style,BGL-trained baseline,0.0073,-0.0432,0.0485


Load the analyses of each set and of detector seeds 1 to 10.

In [26]:
sets_an = {}
seeds_an = {}
for app in APPS:
    sets_an[app] = [load(AN / "sets" / f"set{s}" / app / "analysis.json") for s in (1, 2, 3)]
    seeds_an[app] = [load(AN / "seeds" / f"seed{s}" / app / "analysis.json") for s in range(1, 11)]

Pooled forfeiture at detector seed 42, per set, and over detector seeds 1 to 10 (table forfeiture_spread).

In [27]:
rows = []
for app in APPS:
    for fam in SF:
        for h, base in BASE.items():
            sv = [a["families"][fam]["tests"][h]["gap"] for a in sets_an[app]]
            dv = [a["families"][fam]["tests"][h]["gap"] for a in seeds_an[app]]
            viable_sets = sum(a["families"][fam]["decisions"][base]["viable_substitute"] for a in sets_an[app])
            viable_seeds = sum(a["families"][fam]["decisions"][base]["viable_substitute"] for a in seeds_an[app])
            rows.append({"application": app, "source_free": SF[fam], "baseline": LABEL[base],
                         "pooled_seed42": analysis[app]["families"][fam]["tests"][h]["gap"],
                         "set1": sv[0], "set2": sv[1], "set3": sv[2],
                         "sets_mean": np.mean(sv), "sets_sd": np.std(sv, ddof=1),
                         "seeds_min": min(dv), "seeds_max": max(dv),
                         "viable_sets": viable_sets, "viable_seeds": viable_seeds})
spread = save(pd.DataFrame(rows), "forfeiture_spread")
spread.round(4)

,application,source_free,baseline,pooled_seed42,set1,set2,set3,sets_mean,sets_sd,seeds_min,seeds_max,viable_sets,viable_seeds
0,orders,Isolation Forest,BGL-trained baseline,-0.0167,-0.0051,-0.0264,-0.0185,-0.0167,0.0108,-0.0175,-0.0047,3,10
1,orders,Isolation Forest,OpenStack-trained baseline,-0.3304,-0.3208,-0.3328,-0.3376,-0.3304,0.0087,-0.8376,-0.3237,3,10
2,orders,Isolation Forest,Similar-application baseline,0.0080,0.0288,-0.0257,0.0200,0.0077,0.0293,0.0007,0.0289,3,10
3,orders,DeepLog-style,BGL-trained baseline,-0.0460,-0.0462,-0.0514,-0.0402,-0.0459,0.0056,-0.0514,-0.0270,3,10
4,orders,DeepLog-style,OpenStack-trained baseline,-0.3597,-0.3619,-0.3578,-0.3593,-0.3597,0.0021,-0.8733,-0.3407,3,10
5,orders,DeepLog-style,Similar-application baseline,-0.0213,-0.0123,-0.0507,-0.0017,-0.0216,0.0258,-0.0334,0.0014,3,10
6,bookings,Isolation Forest,BGL-trained baseline,-0.0154,-0.0125,-0.0105,-0.0228,-0.0153,0.0066,-0.0195,-0.0075,3,10
7,bookings,Isolation Forest,OpenStack-trained baseline,-0.3264,-0.3251,-0.3175,-0.3362,-0.3263,0.0094,-0.8314,-0.3205,3,10
8,bookings,Isolation Forest,Similar-application baseline,-0.0064,-0.0025,-0.0096,-0.0071,-0.0064,0.0036,-0.0023,0.0166,3,10
9,bookings,DeepLog-style,BGL-trained baseline,-0.0161,-0.0080,-0.0187,-0.0211,-0.0159,0.0070,-0.0487,-0.0166,3,10


Figure seed_check.pdf: range over detector seeds 1 to 10 (bar) and seed 42 (diamond).

In [28]:
fig, axes = plt.subplots(1, 2, figsize=(8, 2.9), sharey=True)
for ax, app in zip(axes, APPS):
    d = spread[spread.application == app].set_index(["source_free", "baseline"])
    y = np.arange(len(pairs))[::-1]
    for yy, pp in zip(y, pairs):
        ax.plot([d.loc[pp, "seeds_min"], d.loc[pp, "seeds_max"]], [yy, yy], color="grey", lw=6, alpha=0.6, solid_capstyle="butt")
        ax.plot(d.loc[pp, "pooled_seed42"], yy, marker="D", ms=5, color="black", ls="")
    ax.axvline(0, color="grey", lw=0.8)
    ax.axvline(MARGIN, color="grey", ls="--", lw=0.8)
    ax.set_yticks(y)
    ax.set_yticklabels([t.replace("\n", " ") for t in PAIR_TICK], fontsize=7)
    ax.set_xlim(min(0, spread.seeds_min.min(), spread.pooled_seed42.min()) - 0.005,
                max(MARGIN, spread.seeds_max.max(), spread.pooled_seed42.max()) + 0.01)
    ax.set_title(app.capitalize())
    ax.set_xlabel("Pooled F1 forfeiture")
handles = [Line2D([], [], color="grey", lw=6, alpha=0.6, label="Range over detector seeds 1 to 10"),
           Line2D([], [], marker="D", ms=5, color="black", ls="", label="Detector seed 42 (primary result)"),
           Line2D([], [], color="grey", ls="--", lw=0.8, label=f"Viability margin {MARGIN:.2f}")]
fig.legend(handles=handles, loc="lower center", ncol=3, fontsize=7, bbox_to_anchor=(0.5, -0.16))
suptitle(fig, "Pooled forfeiture at detector seed 42 against detector seeds 1 to 10")
fig.savefig(FIG / "seed_check.pdf")
plt.close(fig)

Where seed 42 lies against the range of seeds 1 to 10.

In [29]:
position = np.where(spread.pooled_seed42 > spread.seeds_max, "above range",
                    np.where(spread.pooled_seed42 < spread.seeds_min, "below range", "inside range"))
view = spread.assign(seed42_position=position)
cols = ["application", "source_free", "baseline", "pooled_seed42", "seeds_min", "seeds_max", "seed42_position", "viable_seeds"]
print(view[cols].round(4).to_string(index=False))
print(f"highest forfeiture over all seeds and pairs: {max(spread.seeds_max.max(), spread.pooled_seed42.max()):.4f}")

application      source_free                     baseline  pooled_seed42  seeds_min  seeds_max seed42_position  viable_seeds
     orders Isolation Forest         BGL-trained baseline        -0.0167    -0.0175    -0.0047    inside range            10
     orders Isolation Forest   OpenStack-trained baseline        -0.3304    -0.8376    -0.3237    inside range            10
     orders Isolation Forest Similar-application baseline         0.0080     0.0007     0.0289    inside range            10
     orders    DeepLog-style         BGL-trained baseline        -0.0460    -0.0514    -0.0270    inside range            10
     orders    DeepLog-style   OpenStack-trained baseline        -0.3597    -0.8733    -0.3407    inside range            10
     orders    DeepLog-style Similar-application baseline        -0.0213    -0.0334     0.0014    inside range            10
   bookings Isolation Forest         BGL-trained baseline        -0.0154    -0.0195    -0.0075    inside range            10


Hypothesis decisions (Holm, alpha 0.05) in the pooled analysis, the three sets and the ten detector seeds (table decision_agreement). A hypothesis with nothing to test is counted as not testable.

In [30]:
def rejected(a, fam, h):
    return a["families"][fam]["tests"]["holm"][h]["reject"]

def count(group, fam, h):
    text = f"{sum(rejected(a, fam, h) is True for a in group)} of {len(group)}"
    untestable = sum(rejected(a, fam, h) is None for a in group)
    if untestable:
        text += f" ({untestable} not testable)"
    return text

rows = []
for app in APPS:
    for fam in SF:
        for h in ["H1", "H2", "H3", "H4", "H5", "H6"]:
            rows.append({"application": app, "family": SF[fam], "hypothesis": h,
                         "rejected_pooled": rejected(analysis[app], fam, h),
                         "rejected_in_sets": count(sets_an[app], fam, h),
                         "rejected_in_seeds": count(seeds_an[app], fam, h)})
decisions = save(pd.DataFrame(rows), "decision_agreement")
decisions

,application,family,hypothesis,rejected_pooled,rejected_in_sets,rejected_in_seeds
0,orders,Isolation Forest,H1,True,1 of 3,7 of 10
1,orders,Isolation Forest,H2,True,3 of 3,10 of 10
2,orders,Isolation Forest,H3,True,3 of 3,8 of 10
3,orders,Isolation Forest,H4,True,3 of 3,10 of 10
4,orders,Isolation Forest,H5,True,3 of 3,10 of 10
5,orders,Isolation Forest,H6,True,2 of 3,10 of 10
6,orders,DeepLog-style,H1,True,1 of 3,8 of 10
7,orders,DeepLog-style,H2,True,3 of 3,10 of 10
8,orders,DeepLog-style,H3,False,1 of 3,1 of 10
9,orders,DeepLog-style,H4,True,3 of 3,10 of 10


5. Recall per injected fault category (table recall_by_category).

In [31]:
rows = []
for app in APPS:
    for arm in ARMS:
        tp = {}
        pos = {}
        for r in RUN_TAGS:
            w = preds[(r, app)]
            y = np.array(w["y_true"])
            p = np.array(w["y_pred"][arm])
            category = np.array(w["window_categories"])
            for c in CATS:
                m = (category == c) & (y == 1)
                tp[c] = tp.get(c, 0) + int((p[m] == 1).sum())
                pos[c] = pos.get(c, 0) + int(m.sum())
        for c in tp:
            rows.append({"application": app, "detector": LABEL[arm], "category": c, "detected": tp[c],
                         "fault_windows": pos[c], "recall": ratio(tp[c], pos[c])})
cat = save(pd.DataFrame(rows), "recall_by_category")
cat.pivot_table(index=["application", "detector"], columns="category", values="recall").round(3)

category                                    config  latency  permission  \
application detector                                                      
bookings    BGL-trained baseline             0.830    0.001       0.807   
            DeepLog-style (source-free)      0.867    0.353       0.851   
            Isolation Forest (source-free)   0.838    0.148       0.817   
            OpenStack-trained baseline       0.830    0.000       0.807   
            Similar-application baseline     0.823    0.074       0.803   
            Threshold alarms                 0.521    0.000       0.527   
orders      BGL-trained baseline             0.831    0.000       0.832   
            DeepLog-style (source-free)      0.893    0.758       0.890   
            Isolation Forest (source-free)   0.838    0.210       0.836   
            OpenStack-trained baseline       0.831    0.000       0.832   
            Similar-application baseline     0.827    0.180       0.832   
            Threshold alarms                 0.514    0.000       0.493   

category                                    resource  timeout  
application detector                                           
bookings    BGL-trained baseline               0.962    0.857  
            DeepLog-style (source-free)        0.962    0.866  
            Isolation Forest (source-free)     0.964    0.865  
            OpenStack-trained baseline         0.015    0.000  
            Similar-application baseline       0.959    0.857  
            Threshold alarms                   0.010    0.000  
orders      BGL-trained baseline               0.975    0.878  
            DeepLog-style (source-free)        0.975    0.897  
            Isolation Forest (source-free)     0.976    0.879  
            OpenStack-trained baseline         0.009    0.000  
            Similar-application baseline       0.971    0.882  
            Threshold alarms                   0.005    0.000

Figure recall_by_category.pdf (Figure 3 of the report).

In [32]:
fig, axes = plt.subplots(1, 2, figsize=(8, 3.0), sharey=True)
for ax, app in zip(axes, APPS):
    d = cat[cat.application == app]
    for i, arm in enumerate(ARMS):
        vals = [d[(d.detector == LABEL[arm]) & (d.category == c)].recall.iloc[0] for c in CATS]
        label = "Keyword alarms" if arm == "threshold" else LABEL[arm]
        ax.bar(np.arange(len(CATS)) + (i - (len(ARMS) - 1) / 2) * 0.13, vals, width=0.13, label=label, color=DET_COLOR[arm])
    ax.set_xticks(range(len(CATS)))
    ax.set_xticklabels(CATS, fontsize=8)
    ax.set_title(app.capitalize())
    ax.set_ylim(0, 1.05)
    ax.set_xlabel("Injected fault category")
axes[0].set_ylabel("Recall (share of fault windows flagged)")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3, fontsize=8, bbox_to_anchor=(0.5, -0.2))
fig.savefig(FIG / "recall_by_category.pdf")
plt.close(fig)

Lowest and highest recall of the learned detectors per category, and the threshold alarms' recall.

In [33]:
learned = cat[cat.detector != LABEL["threshold"]]
print(learned.groupby(["application", "category"]).recall.agg(["min", "max"]).round(3))
print(cat[cat.detector == LABEL["threshold"]].pivot_table(index="application", columns="category", values="recall").round(3))

                          min    max
application category                
bookings    config      0.823  0.867
            latency     0.000  0.353
            permission  0.803  0.851
            resource    0.015  0.964
            timeout     0.000  0.866
orders      config      0.827  0.893
            latency     0.000  0.758
            permission  0.832  0.890
            resource    0.009  0.976
            timeout     0.000  0.897
category     config  latency  permission  resource  timeout
application                                                
bookings      0.521      0.0       0.527     0.010      0.0
orders        0.514      0.0       0.493     0.005      0.0


6. False alarms on held-out normal windows and on elasticity windows (table false_alarms).

In [34]:
rows = []
for app in APPS:
    for arm in ARMS:
        neg = 0
        fp = 0
        ctl = 0
        ctl_fp = 0
        for r in RUN_TAGS:
            w = preds[(r, app)]
            y = np.array(w["y_true"])
            p = np.array(w["y_pred"][arm])
            c = np.array(w["control_windows"])
            neg += int((y == 0).sum())
            fp += int(((p == 1) & (y == 0)).sum())
            ctl += int(c.sum())
            ctl_fp += int(((p == 1) & c).sum())
        rows.append({"application": app, "detector": LABEL[arm], "false_alarms_heldout": fp, "normal_windows": neg,
                     "rate_heldout": ratio(fp, neg), "false_alarms_elasticity": ctl_fp, "elasticity_windows": ctl,
                     "rate_elasticity": ratio(ctl_fp, ctl)})
far = save(pd.DataFrame(rows), "false_alarms")
far.round(4)

,application,detector,false_alarms_heldout,normal_windows,rate_heldout,false_alarms_elasticity,elasticity_windows,rate_elasticity
0,orders,Isolation Forest (source-free),121,9469,0.0128,61,816,0.0748
1,orders,DeepLog-style (source-free),597,9469,0.0630,144,816,0.1765
2,orders,BGL-trained baseline,3,9469,0.0003,0,816,0.0000
3,orders,OpenStack-trained baseline,0,9469,0.0000,0,816,0.0000
4,orders,Similar-application baseline,3,9469,0.0003,1,816,0.0012
5,orders,Threshold alarms,0,9469,0.0000,0,816,0.0000
6,bookings,Isolation Forest (source-free),82,9477,0.0087,60,816,0.0735
7,bookings,DeepLog-style (source-free),386,9477,0.0407,118,816,0.1446
8,bookings,BGL-trained baseline,2,9477,0.0002,0,816,0.0000
9,bookings,OpenStack-trained baseline,0,9477,0.0000,0,816,0.0000


Figure false_alarms.pdf: dashed line is the 1% target.

In [35]:
fig, axes = plt.subplots(1, 2, figsize=(8, 3.2), sharey=True)
for ax, app in zip(axes, APPS):
    d = far[far.application == app]
    x = np.arange(len(ARMS))
    b1 = ax.bar(x - 0.2, d.rate_heldout, 0.4, label="Held-out normal windows")
    b2 = ax.bar(x + 0.2, d.rate_elasticity, 0.4, label="Elasticity windows")
    for bars in (b1, b2):
        for b in bars:
            ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.003, f"{b.get_height():.3f}",
                    ha="center", va="bottom", fontsize=6, rotation=90)
    ax.axhline(TARGET, color="grey", ls="--", lw=0.8, label="1% target set on the training window")
    ax.set_xticks(x)
    ax.set_xticklabels(SHORT, fontsize=7)
    ax.set_title(app.capitalize())
    ax.set_ylim(0, max(far.rate_heldout.max(), far.rate_elasticity.max(), TARGET) * 1.25)
axes[0].set_ylabel("False-alarm rate (share of\nfault-free windows)")
h, l = axes[0].get_legend_handles_labels()
fig.legend(h, l, loc="lower center", ncol=3, fontsize=7, bbox_to_anchor=(0.5, -0.1))
suptitle(fig, "False-alarm rate on fault-free windows (nine runs pooled)")
fig.savefig(FIG / "false_alarms.pdf")
plt.close(fig)

False-alarm rates of the two source-free detectors against the target.

In [36]:
for app in APPS:
    d = far[far.application == app].set_index("detector")
    parts = []
    for i, a in enumerate(ARMS[:2]):
        rate_h = d.loc[LABEL[a], "rate_heldout"]
        rate_e = d.loc[LABEL[a], "rate_elasticity"]
        parts.append(f"{flat(i)} held-out {rate_h:.3f}, elasticity {rate_e:.3f} ({rate_e / TARGET:.1f} x target)")
    print(f"{app}: " + "; ".join(parts))

orders: Isolation Forest held-out 0.013, elasticity 0.075 (7.5 x target); DeepLog-style held-out 0.063, elasticity 0.176 (17.6 x target)
bookings: Isolation Forest held-out 0.009, elasticity 0.074 (7.4 x target); DeepLog-style held-out 0.041, elasticity 0.145 (14.5 x target)


7. Lag-1 autocorrelation of the window labels and of the Isolation Forest's errors, mean over the runs.

In [37]:
rows = []
for app in APPS:
    for r in RUN_TAGS:
        w = preds[(r, app)]
        y = np.array(w["y_true"], float)
        e = (np.array(w["y_pred"]["source_free"]) != y).astype(float)
        lag1_errors = np.corrcoef(e[:-1], e[1:])[0, 1] if e.std() > 0 else np.nan
        rows.append({"application": app, "run": r, "lag1_labels": np.corrcoef(y[:-1], y[1:])[0, 1],
                     "lag1_isolation_forest_errors": lag1_errors})
ac = pd.DataFrame(rows)
print(ac.groupby("application")[["lag1_labels", "lag1_isolation_forest_errors"]].mean().round(3))

             lag1_labels  lag1_isolation_forest_errors
application                                           
bookings           0.036                         0.129
orders             0.035                         0.115


Forfeiture with runs as the unit: t-interval, Cohen's d, Shapiro-Wilk and Wilcoxon tests (table forfeiture_run_level).

In [38]:
def rank_biserial(d):
    # matched-pairs rank-biserial correlation of the Wilcoxon signed-rank test; NaN when every value is 0
    nz = d[d != 0]
    r = stats.rankdata(np.abs(nz))
    if nz.size == 0:
        return np.nan
    return (r[nz > 0].sum() - r[nz < 0].sum()) / r.sum()

rows = []
for app in APPS:
    for fam in SF:
        for h, base in BASE.items():
            d = np.array([results[(r, app)]["summaries"][base]["f1"] - results[(r, app)]["summaries"][fam]["f1"] for r in RUN_TAGS])
            lo, hi = stats.t.interval(0.95, len(d) - 1, loc=d.mean(), scale=stats.sem(d))
            sd = d.std(ddof=1)
            rows.append({"application": app, "source_free": SF[fam], "baseline": LABEL[base], "runs": len(d),
                         "mean": d.mean(), "sd": sd, "min": d.min(), "max": d.max(), "t_ci_low": lo, "t_ci_high": hi,
                         "cohens_dz": d.mean() / sd if sd > 0 else np.nan,
                         "shapiro_p": stats.shapiro(d).pvalue, "wilcoxon_p": stats.wilcoxon(d).pvalue,
                         "rank_biserial": rank_biserial(d), "runs_above_margin": int((d > MARGIN).sum())})
runlevel = save(pd.DataFrame(rows), "forfeiture_run_level")
check = per_run_forf.groupby(["application", "source_free", "baseline"], sort=False).forfeiture.mean().to_numpy()
assert np.allclose(check, runlevel["mean"].to_numpy())
runlevel.round(4)

,application,source_free,baseline,runs,mean,sd,min,max,t_ci_low,t_ci_high,cohens_dz,shapiro_p,wilcoxon_p,rank_biserial,runs_above_margin
0,orders,Isolation Forest,BGL-trained baseline,9,-0.0160,0.0245,-0.0643,0.0151,-0.0348,0.0028,-0.6553,0.3947,0.0977,-0.6444,0
1,orders,Isolation Forest,OpenStack-trained baseline,9,-0.3298,0.0272,-0.3703,-0.2893,-0.3507,-0.3090,-12.1451,0.5111,0.0039,-1.0000,0
2,orders,Isolation Forest,Similar-application baseline,9,0.0081,0.0355,-0.0665,0.0529,-0.0192,0.0353,0.2282,0.2682,0.3594,0.3778,0
3,orders,DeepLog-style,BGL-trained baseline,9,-0.0460,0.0128,-0.0664,-0.0251,-0.0559,-0.0362,-3.5965,0.8430,0.0039,-1.0000,0
4,orders,DeepLog-style,OpenStack-trained baseline,9,-0.3598,0.0072,-0.3688,-0.3488,-0.3653,-0.3543,-50.2121,0.4075,0.0039,-1.0000,0
5,orders,DeepLog-style,Similar-application baseline,9,-0.0219,0.0299,-0.0622,0.0213,-0.0449,0.0011,-0.7324,0.5181,0.0977,-0.6444,0
6,bookings,Isolation Forest,BGL-trained baseline,9,-0.0153,0.0127,-0.0363,0.0001,-0.0250,-0.0055,-1.2036,0.3515,0.0078,-0.9556,0
7,bookings,Isolation Forest,OpenStack-trained baseline,9,-0.3262,0.0164,-0.3519,-0.3081,-0.3388,-0.3137,-19.9309,0.1477,0.0039,-1.0000,0
8,bookings,Isolation Forest,Similar-application baseline,9,-0.0065,0.0190,-0.0363,0.0294,-0.0211,0.0080,-0.3448,0.8898,0.3008,-0.4222,0
9,bookings,DeepLog-style,BGL-trained baseline,9,-0.0142,0.0322,-0.0614,0.0256,-0.0390,0.0106,-0.4402,0.4565,0.2500,-0.4667,0


8. Post hoc pairwise tests for H4 (recall per category) and H5 (elasticity false alarms), Holm-adjusted (table posthoc_h4_h5). An untestable pair is N/A.

In [39]:
def holm(p):
    # the analysis's Holm adjustment over the testable pairs only; an untestable pair (NaN) stays NaN
    adj = holm_family(dict(enumerate(p)), CFG["analysis"]["alpha"])
    return [np.nan if adj[i]["adjusted_p"] is None else adj[i]["adjusted_p"] for i in range(len(p))]

rows = []
for app in APPS:
    for arm in SF:
        d = cat[(cat.application == app) & (cat.detector == LABEL[arm])].set_index("category")
        present = [c for c in CATS if d.loc[c, "fault_windows"] > 0]
        table = np.array([[d.loc[c, "detected"], d.loc[c, "fault_windows"] - d.loc[c, "detected"]] for c in present])
        chi2, _, _, expected = stats.chi2_contingency(table)
        v = np.sqrt(chi2 / table.sum())
        cpairs = [(a, b) for i, a in enumerate(present) for b in present[i + 1:]]
        ps = []
        for a, b in cpairs:
            t = table[[present.index(a), present.index(b)]]
            # a pair whose two categories are all detected or all missed cannot be tested
            ps.append(stats.chi2_contingency(t)[1] if (t.sum(axis=0) > 0).all() else np.nan)
        for (a, b), p, q in zip(cpairs, ps, holm(ps)):
            rows.append({"application": app, "detector": SF[arm], "test": "H4 recall", "pair": f"{a} vs {b}",
                         "difference": d.loc[a, "recall"] - d.loc[b, "recall"], "p": p, "holm_p": q,
                         "cramers_v_overall": v, "min_expected_overall": expected.min()})
for app in APPS:
    alarms = {}
    for arm in ARMS:
        parts = [np.array(preds[(r, app)]["y_pred"][arm])[np.array(preds[(r, app)]["control_windows"])] for r in RUN_TAGS]
        alarms[arm] = np.concatenate(parts)
    apairs = [(a, b) for i, a in enumerate(ARMS) for b in ARMS[i + 1:]]
    ps = []
    diffs = []
    for a, b in apairs:
        n01 = int(((alarms[a] == 1) & (alarms[b] == 0)).sum())
        n10 = int(((alarms[a] == 0) & (alarms[b] == 1)).sum())
        # a pair that never disagrees cannot be tested: its p-value is N/A
        ps.append(stats.binomtest(n01, n01 + n10, 0.5).pvalue if n01 + n10 else np.nan)
        diffs.append(alarms[a].mean() - alarms[b].mean())
    for (a, b), p, q, dd in zip(apairs, ps, holm(ps), diffs):
        rows.append({"application": app, "detector": "all arms", "test": "H5 elasticity false alarms",
                     "pair": f"{LABEL[a]} vs {LABEL[b]}", "difference": dd, "p": p, "holm_p": q,
                     "cramers_v_overall": np.nan, "min_expected_overall": np.nan})
posthoc = save(pd.DataFrame(rows), "posthoc_h4_h5")
posthoc.round(4)

,application,detector,test,pair,difference,p,holm_p,cramers_v_overall,min_expected_overall
0,orders,Isolation Forest,H4 recall,permission vs config,-0.0025,0.9441,0.9441,0.6332,188.4041
1,orders,Isolation Forest,H4 recall,permission vs timeout,-0.0435,0.0180,0.0540,0.6332,188.4041
2,orders,Isolation Forest,H4 recall,permission vs resource,-0.1407,0.0000,0.0000,0.6332,188.4041
3,orders,Isolation Forest,H4 recall,permission vs latency,0.6258,0.0000,0.0000,0.6332,188.4041
4,orders,Isolation Forest,H4 recall,config vs timeout,-0.0410,0.0261,0.0540,0.6332,188.4041
...,...,...,...,...,...,...,...,...,...
65,bookings,all arms,H5 elasticity false alarms,BGL-trained baseline vs Similar-application ba...,0.0000,NaN,NaN,NaN,NaN
66,bookings,all arms,H5 elasticity false alarms,BGL-trained baseline vs Threshold alarms,0.0000,NaN,NaN,NaN,NaN
67,bookings,all arms,H5 elasticity false alarms,OpenStack-trained baseline vs Similar-applicat...,0.0000,NaN,NaN,NaN,NaN
68,bookings,all arms,H5 elasticity false alarms,OpenStack-trained baseline vs Threshold alarms,0.0000,NaN,NaN,NaN,NaN


9. Error text: a window with an ERROR line or an error, permission, config, timeout or resource keyword. Mark every window with or without error text.

In [40]:
RULE = {}
for app in APPS:
    for r in RUN_TAGS:
        w = preds[(r, app)]
        ev = [e for e in events_from_jsonl(RUNS / r / app / "raw" / "events.jsonl") if e.split == "eval"]
        features = lexical_window_features(ev, np.array(w["event_indices"]), WINDOW)
        RULE[(r, app)] = (features.sum(axis=1) > 0).astype(int)

Each baseline's alarms against the rule alarm on error text (table keyword_rule_equivalence).

In [41]:
rows = []
for app in APPS:
    diff_bgl = 0
    diff_sim = 0
    n = 0
    rule_tp = 0
    rule_fp = 0
    for r in RUN_TAGS:
        w = preds[(r, app)]
        rule = RULE[(r, app)]
        y = np.array(w["y_true"])
        n += len(y)
        diff_bgl += int((rule != np.array(w["y_pred"]["transfer"])).sum())
        diff_sim += int((rule != np.array(w["y_pred"]["transfer_similar"])).sum())
        rule_tp += int(((rule == 1) & (y == 1)).sum())
        rule_fp += int(((rule == 1) & (y == 0)).sum())
    pos = analysis[app]["pooled"]["transfer"]["support_pos"]
    prec = ratio(rule_tp, rule_tp + rule_fp)
    rec = ratio(rule_tp, pos)
    rows.append({"application": app, "windows": n, "bgl_differs_from_rule": diff_bgl, "similar_differs_from_rule": diff_sim,
                 "rule_false_alarms": rule_fp, "rule_f1": ratio(2 * prec * rec, prec + rec)})
keyword = save(pd.DataFrame(rows), "keyword_rule_equivalence")
keyword.round(4)

,application,windows,bgl_differs_from_rule,similar_differs_from_rule,rule_false_alarms,rule_f1
0,orders,13438,0,180,3,0.8219
1,bookings,13442,0,71,2,0.8119


Fault windows without error text, and how many each detector flagged (table fault_windows_without_error_text).

In [42]:
rows = []
for app in APPS:
    nt = {arm: 0 for arm in ARMS}
    nt_n = 0
    for r in RUN_TAGS:
        w = preds[(r, app)]
        y = np.array(w["y_true"])
        m = (y == 1) & (RULE[(r, app)] == 0)
        nt_n += int(m.sum())
        for arm in ARMS:
            nt[arm] += int((np.array(w["y_pred"][arm])[m] == 1).sum())
    pos = analysis[app]["pooled"]["transfer"]["support_pos"]
    row = {"application": app, "fault_windows_without_error_text": nt_n, "share_of_fault_windows": ratio(nt_n, pos)}
    for arm in ARMS:
        row[f"detected_by_{arm}"] = nt[arm]
    rows.append(row)
notext = save(pd.DataFrame(rows), "fault_windows_without_error_text")
notext.round(4)

,application,fault_windows_without_error_text,share_of_fault_windows,detected_by_source_free,detected_by_source_free_deeplog,detected_by_transfer,detected_by_transfer_openstack,detected_by_transfer_similar,detected_by_threshold
0,orders,1198,0.3018,182,729,0,0,160,0
1,bookings,1254,0.3163,144,363,0,0,60,0


Recall on fault windows with and without error text (table fault_windows_by_error_text).

In [43]:
rows = []
for app in APPS:
    for arm in ARMS:
        cnt = {"with": [0, 0], "without": [0, 0]}
        for r in RUN_TAGS:
            w = preds[(r, app)]
            y = np.array(w["y_true"])
            p = np.array(w["y_pred"][arm])
            rule = RULE[(r, app)]
            for k, m in (("with", (y == 1) & (rule == 1)), ("without", (y == 1) & (rule == 0))):
                cnt[k][0] += int((p[m] == 1).sum())
                cnt[k][1] += int(m.sum())
        for k in ("with", "without"):
            rows.append({"application": app, "detector": LABEL[arm], "error_text": k, "fault_windows": cnt[k][1],
                         "detected": cnt[k][0], "recall": ratio(cnt[k][0], cnt[k][1])})
bytext = save(pd.DataFrame(rows), "fault_windows_by_error_text")
bytext.pivot_table(index=["application", "detector"], columns="error_text", values="recall").round(3)

error_text                                   with  without
application detector                                      
bookings    BGL-trained baseline            1.000    0.000
            DeepLog-style (source-free)     1.000    0.289
            Isolation Forest (source-free)  1.000    0.115
            OpenStack-trained baseline      0.489    0.000
            Similar-application baseline    0.996    0.048
            Threshold alarms                0.313    0.000
orders      BGL-trained baseline            1.000    0.000
            DeepLog-style (source-free)     1.000    0.609
            Isolation Forest (source-free)  1.000    0.152
            OpenStack-trained baseline      0.488    0.000
            Similar-application baseline    0.994    0.134
            Threshold alarms                0.296    0.000

Check: the counts agree with the pooled true positives and with table fault_windows_without_error_text.

In [44]:
for app in APPS:
    for arm in ARMS:
        d = bytext[(bytext.application == app) & (bytext.detector == LABEL[arm])]
        assert d.detected.sum() == analysis[app]["pooled"][arm]["tp"]
        assert d[d.error_text == "without"].detected.iloc[0] == notext.set_index("application").loc[app, f"detected_by_{arm}"]
print("error text check passed")

error text check passed


Figure error_text_detection.pdf.

In [45]:
fig, axes = plt.subplots(1, 2, figsize=(8, 3.3), sharey=True)
for ax, app in zip(axes, APPS):
    d = bytext[bytext.application == app]
    x = np.arange(len(ARMS))
    group_sizes = []
    for k, off, col in (("with", -0.2, "#4d4d4d"), ("without", 0.2, "#e08214")):
        dd = d[d.error_text == k].set_index("detector").loc[[LABEL[a] for a in ARMS]]
        bars = ax.bar(x + off, dd.recall, 0.4, color=col)
        for b, nflag in zip(bars, dd.detected):
            ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.015, f"{nflag:,}",
                    ha="center", va="bottom", fontsize=6, rotation=90)
        group_sizes.append(f"{dd.fault_windows.iloc[0]:,}")
    ax.set_xticks(x)
    ax.set_xticklabels(SHORT, fontsize=7)
    ax.set_ylim(0, 1.15)
    ax.set_title(f"{app.capitalize()}\n({group_sizes[0]} fault windows with error text, {group_sizes[1]} without)", fontsize=8)
fig.legend(handles=[Patch(color="#4d4d4d", label="Fault windows with error text"),
                    Patch(color="#e08214", label="Fault windows without error text")],
           loc="lower center", ncol=2, fontsize=7, bbox_to_anchor=(0.5, -0.08))
axes[0].set_ylabel("Share of the group flagged\n(number flagged above bar)")
suptitle(fig, "Fault windows with and without error text flagged by each detector (supplementary)")
fig.savefig(FIG / "error_text_detection.pdf")
plt.close(fig)

Fault segments flagged only by the source-free detector or only by the BGL-trained baseline, exact McNemar test (table fault_segment_detection).

In [46]:
rows = []
for app in APPS:
    for fam in SF:
        sm = analysis[app]["families"][fam]["tests"]["H1"]["segment_mcnemar"]
        row = {"application": app, "family": SF[fam]}
        for k, v in sm.items():
            if not isinstance(v, (dict, list)):
                row[k] = v
        rows.append(row)
save(pd.DataFrame(rows), "fault_segment_detection")

,application,family,n_segments,only_first_right,only_second_right,p
0,orders,Isolation Forest,2701,0,170,1.336382e-51
1,orders,DeepLog-style,2701,0,540,5.556897e-163
2,bookings,Isolation Forest,2697,0,120,1.504633e-36
3,bookings,DeepLog-style,2697,0,247,8.843437e-75


10. Ablations: pooled measures of the parts of the Isolation Forest detector (table ablation_pooled).

In [47]:
ABLATION_ARMS = ["source_free", "source_free_forest", "novelty_gate", "source_free_source_features", "transfer"]
rows = []
for app in APPS:
    for arm in ABLATION_ARMS:
        p = analysis[app]["pooled"][arm]
        rows.append({"application": app, "detector": LABEL[arm], "precision": p["precision"], "recall": p["recall"],
                     "f1": p["f1"], "mcc": p["mcc"]})
ablation = save(pd.DataFrame(rows), "ablation_pooled")
ablation.round(4)

,application,detector,precision,recall,f1,mcc
0,orders,Isolation Forest (source-free),0.9606,0.7440,0.8386,0.7942
1,orders,Isolation Forest alone,0.9392,0.4434,0.6024,0.5681
2,orders,Novelty gate alone,0.9975,0.6987,0.8218,0.7860
3,orders,Source-free detector on the BGL baseline's fea...,0.9989,0.6982,0.8219,0.7866
4,orders,BGL-trained baseline,0.9989,0.6982,0.8219,0.7866
5,bookings,Isolation Forest (source-free),0.9721,0.7201,0.8273,0.7851
6,bookings,Isolation Forest alone,0.9577,0.4230,0.5868,0.5625
7,bookings,Novelty gate alone,0.9971,0.6840,0.8114,0.7754
8,bookings,Source-free detector on the BGL baseline's fea...,0.9993,0.6837,0.8119,0.7766
9,bookings,BGL-trained baseline,0.9993,0.6837,0.8119,0.7766


Where the combined detector's alarms come from, and the feature-space check (table novelty_gate_alarms).

In [48]:
rows = []
for app in APPS:
    g = analysis[app]["supplementary"]["novelty_gate"]
    f = analysis[app]["supplementary"]["feature_space"]
    rows.append({"application": app, "alarms": g["alarms"], "gate_only": g["alarms_from_the_gate_only"],
                 "forest_only": g["alarms_from_the_forest_only"], "both": g["alarms_from_both"],
                 "share_gate_only": ratio(g["alarms_from_the_gate_only"], g["alarms"]),
                 "feature_space_gap": f["gap"], "feature_space_ci_low": f["ci_low"], "feature_space_ci_high": f["ci_high"]})
gate = save(pd.DataFrame(rows), "novelty_gate_alarms")
gate.round(4)

,application,alarms,gate_only,forest_only,both,share_gate_only,feature_space_gap,feature_space_ci_low,feature_space_ci_high
0,orders,3074,1200,294,1580,0.3904,0.0,0.0,0.0
1,bookings,2937,1186,217,1534,0.4038,0.0,0.0,0.0


Figure ablations.pdf.

In [49]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 3.2), gridspec_kw={"width_ratios": [2, 1.2]})
x = np.arange(len(APPS))
for k, arm in enumerate(ABLATION_ARMS):
    vals = [ablation[(ablation.application == app) & (ablation.detector == LABEL[arm])].f1.iloc[0] for app in APPS]
    bars = ax1.bar(x + (k - 2) * 0.16, vals, 0.16, color=DET_COLOR[arm], label=LABEL[arm])
    for b in bars:
        ax1.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.01, f"{b.get_height():.2f}",
                 ha="center", va="bottom", fontsize=6, rotation=90)
ax1.set_xticks(x)
ax1.set_xticklabels([a.capitalize() for a in APPS])
ax1.set_ylim(0, 1.15)
ax1.set_ylabel("Pooled F1")
ax1.legend(fontsize=6.5, loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2, frameon=False)
y = np.arange(len(APPS))
left = np.zeros(len(APPS))
for col, label, c in (("gate_only", "Gate only", "#fdae6b"), ("both", "Both", "tab:blue"), ("forest_only", "Forest only", "#9ecae1")):
    share = (gate[col] / gate.alarms).to_numpy()
    ax2.barh(y, share, left=left, color=c, label=label)
    for yy, l, s in zip(y, left, share):
        if s > 0.06:
            ax2.text(l + s / 2, yy, f"{s:.0%}", ha="center", va="center", fontsize=7)
    left += share
ax2.set_yticks(y)
ax2.set_yticklabels([a.capitalize() for a in APPS])
ax2.set_xlim(0, 1)
ax2.invert_yaxis()
ax2.set_xlabel("Share of the combined detector's alarms")
ax2.legend(fontsize=7, loc="upper center", bbox_to_anchor=(0.5, -0.25), ncol=3, frameon=False)
suptitle(fig, "Ablations of the source-free Isolation Forest (supplementary)")
fig.tight_layout()
fig.savefig(FIG / "ablations.pdf")
plt.close(fig)

11. The BGL-trained baseline on the LogHub benchmark pairs, beside the published ELFA-Log values for scale only (table calibration_vs_elfa_log). The baseline's F1 is 0 to 1, ELFA-Log's in percent.

In [50]:
c = load(ROOT / "results" / "calibration" / "calibration.json")
rows = []
for pair, res in c["results"].items():
    ref = res["elfa_log_reference"]
    m = res["baseline"]
    rows.append({"pair": pair, "f1": m.get("f1"), "mcc": m.get("mcc"), "elfa_log_f1": ref.get("f1"), "elfa_log_mcc": ref.get("mcc")})
calib = save(pd.DataFrame(rows), "calibration_vs_elfa_log")
calib

,pair,f1,mcc,elfa_log_f1,elfa_log_mcc
0,BGL->HDFS,0.008343,-0.014268,71.83,0.674
1,BGL->OpenStack,0.002167,0.022788,58.43,0.054


Figure calibration_benchmarks.pdf: n/a marks an MCC that is undefined because the baseline raised no alarm.

In [51]:
pairs_c = list(c["results"])
x = np.arange(len(pairs_c))
fig, axes = plt.subplots(1, 2, figsize=(6, 3))
for ax, (metric, ylab, scale) in zip(axes, (("f1", "F1 (%)", 100), ("mcc", "MCC", 1))):
    pad = 0.01 * (100 if metric == "f1" else 1)
    for xi, p in zip(x, pairs_c):
        v = calib[calib.pair == p][metric].iloc[0]
        if v is None or pd.isna(v):
            ax.text(xi - 0.15, pad, "n/a", ha="center", va="bottom", fontsize=7, rotation=90)
        else:
            ax.bar(xi - 0.15, v * scale, 0.3, color=DET_COLOR["transfer"])
            text = f"{v * scale:.2f}" if metric == "f1" else f"{v:.3f}"
            ax.text(xi - 0.15, max(v * scale, 0) + pad, text, ha="center", va="bottom", fontsize=7, rotation=90)
        ref = calib[calib.pair == p][f"elfa_log_{metric}"].iloc[0]
        ax.bar(xi + 0.15, ref, 0.3, color="lightgrey", hatch="///", edgecolor="grey")
        text = f"{ref:.2f}" if metric == "f1" else f"{ref:.3f}"
        ax.text(xi + 0.15, ref + pad, text, ha="center", va="bottom", fontsize=7, rotation=90)
    ax.set_xticks(x)
    ax.set_xticklabels([p.replace("->", " to ") for p in pairs_c])
    ax.set_ylabel(ylab)
    ax.axhline(0, color="grey", lw=0.8)
    ax.set_ylim(-0.05 if metric == "mcc" else 0, 100 if metric == "f1" else 1.0)
axes[0].set_title("F1 on the benchmark pair")
axes[1].set_title("MCC on the benchmark pair")
handles = [Patch(color=DET_COLOR["transfer"], label="BGL-trained baseline"),
           Patch(facecolor="lightgrey", hatch="///", edgecolor="grey", label="ELFA-Log, published value")]
fig.legend(handles=handles, loc="lower center", ncol=2, fontsize=7, bbox_to_anchor=(0.5, -0.1))
fig.savefig(FIG / "calibration_benchmarks.pdf")
plt.close(fig)
for p in pairs_c:
    d = calib[calib.pair == p].iloc[0]
    print(f"{p}: baseline F1 (%) {100 * d.f1:.2f}; ELFA-Log published {d.elfa_log_f1:.2f}")

BGL->HDFS: baseline F1 (%) 0.83; ELFA-Log published 71.83
BGL->OpenStack: baseline F1 (%) 0.22; ELFA-Log published 58.43


12. Pooled F1 with windows of 10, 20 and 50 lines, and with 25%, 50%, 75% and all of the training part (table sensitivity_f1).

In [52]:
rows = []
for app in APPS:
    variants = [("window", 20, analysis[app])]
    for w in (10, 50):
        variants.append(("window", w, load(AN / "windows" / f"window{w}" / app / "analysis.json")))
    variants.append(("training share", 1.0, analysis[app]))
    for f in (0.25, 0.5, 0.75):
        variants.append(("training share", f, load(AN / "training-share" / f"share{f}" / app / "analysis.json")))
    for kind, value, a in variants:
        row = {"application": app, "setting": kind, "value": value}
        for arm in ARMS:
            row[LABEL[arm]] = a["pooled"][arm]["f1"]
        rows.append(row)
sensitivity = save(pd.DataFrame(rows).sort_values(["application", "setting", "value"]), "sensitivity_f1")
sensitivity.round(4)

,application,setting,value,Isolation Forest (source-free),DeepLog-style (source-free),BGL-trained baseline,OpenStack-trained baseline,Similar-application baseline,Threshold alarms
11,bookings,training share,0.25,0.8259,0.8300,0.8119,0.5009,0.8209,0.3527
12,bookings,training share,0.50,0.8318,0.8505,0.8119,0.5009,0.8209,0.3527
13,bookings,training share,0.75,0.8285,0.8321,0.8119,0.5009,0.8209,0.3527
10,bookings,training share,1.00,0.8273,0.8280,0.8119,0.5009,0.8209,0.3527
8,bookings,window,10.00,0.8130,0.7813,0.7736,0.3207,0.7277,0.2030
7,bookings,window,20.00,0.8273,0.8280,0.8119,0.5009,0.8209,0.3527
9,bookings,window,50.00,0.8578,0.8799,0.4655,0.5385,0.9337,0.4684
4,orders,training share,0.25,0.8592,0.8694,0.8219,0.5082,0.8465,0.3421
5,orders,training share,0.50,0.8525,0.8661,0.8219,0.5082,0.8465,0.3421
6,orders,training share,0.75,0.8401,0.8609,0.8219,0.5082,0.8465,0.3421


13. Recall per injected delay, and false alarms on elasticity windows with and without a new execution environment (table latency_and_cold_start).

In [53]:
rows = []
for app in APPS:
    sup = analysis[app]["supplementary"]
    for ms, v in sup["latency_recall_by_delay"].items():
        row = {"application": app, "measure": f"recall, {int(ms) // 1000} s delay", "windows": v["windows"]}
        for a in ARMS:
            row[LABEL[a]] = v["recall"][a]
        rows.append(row)
    for part, v in sup["elasticity_false_alarms"].items():
        row = {"application": app, "measure": "false-alarm rate, elasticity " + part.replace("_", " "), "windows": v["windows"]}
        for a in ARMS:
            row[LABEL[a]] = v["false_alarm_rate"][a]
        rows.append(row)
latency_cold = save(pd.DataFrame(rows), "latency_and_cold_start")
latency_cold.round(3)

,application,measure,windows,Isolation Forest (source-free),DeepLog-style (source-free),BGL-trained baseline,OpenStack-trained baseline,Similar-application baseline,Threshold alarms
0,orders,"recall, 4 s delay",415,0.210,0.754,0.000,0.0,0.169,0.0
1,orders,"recall, 6 s delay",400,0.210,0.762,0.000,0.0,0.192,0.0
2,orders,"false-alarm rate, elasticity with new environment",78,0.577,0.756,0.000,0.0,0.013,0.0
3,orders,"false-alarm rate, elasticity without new envir...",738,0.022,0.115,0.000,0.0,0.000,0.0
4,bookings,"recall, 4 s delay",414,0.157,0.355,0.000,0.0,0.085,0.0
5,bookings,"recall, 6 s delay",411,0.139,0.350,0.002,0.0,0.063,0.0
6,bookings,"false-alarm rate, elasticity with new environment",69,0.609,0.681,0.000,0.0,0.000,0.0
7,bookings,"false-alarm rate, elasticity without new envir...",747,0.024,0.095,0.000,0.0,0.000,0.0


Training and scoring time per detector in seconds, mean over the runs (table run_time).

In [54]:
rows = []
for app in APPS:
    for arm in ARMS + ABLATIONS:
        sec = [results[(r, app)]["detector_meta"][arm]["seconds"] for r in RUN_TAGS]
        rows.append({"application": app, "detector": LABEL[arm], "fit_seconds": np.mean([s["fit"] for s in sec]),
                     "predict_seconds": np.mean([s["predict"] for s in sec])})
timing = save(pd.DataFrame(rows), "run_time")
timing.round(2)

,application,detector,fit_seconds,predict_seconds
0,orders,Isolation Forest (source-free),0.31,0.57
1,orders,DeepLog-style (source-free),12.76,1.21
2,orders,BGL-trained baseline,0.29,0.57
3,orders,OpenStack-trained baseline,3.32,0.56
4,orders,Similar-application baseline,1.52,0.58
5,orders,Threshold alarms,0.00,0.07
6,orders,Isolation Forest alone,0.31,0.57
7,orders,Novelty gate alone,0.16,0.56
8,orders,Source-free detector on the BGL baseline's fea...,0.29,0.56
9,bookings,Isolation Forest (source-free),0.33,0.61


14. Alarm threshold, calibration windows and held-out false-alarm rate of each learned detector over the 18 run and application pairs (table alarm_calibration).

In [55]:
rows = []
for arm in [a for a in ARMS + ABLATIONS if a != "threshold"]:
    th = []
    cw = []
    fa = []
    for r in RUN_TAGS:
        for app in APPS:
            m = results[(r, app)]["detector_meta"][arm]
            th.append(m.get("decision_threshold", m.get("top_g")))
            cw.append(m["n_calib_windows"])
            fa.append(results[(r, app)]["summaries"][arm]["false_alarm_rate"])
    th = [t for t in th if t is not None]
    rows.append({"detector": LABEL[arm],
                 "threshold_low": min(th) if th else float("nan"), "threshold_high": max(th) if th else float("nan"),
                 "calibration_windows_low": min(cw), "calibration_windows_high": max(cw),
                 "false_alarm_rate_low": min(fa), "false_alarm_rate_high": max(fa),
                 "pairs_above_target": sum(f > TARGET for f in fa)})
calibration_table = save(pd.DataFrame(rows), "alarm_calibration")
calibration_table.round(4)

,detector,threshold_low,threshold_high,calibration_windows_low,calibration_windows_high,false_alarm_rate_low,false_alarm_rate_high,pairs_above_target
0,Isolation Forest (source-free),0.1626,0.2375,153,156,0.0009,0.0239,12
1,DeepLog-style (source-free),2.0000,2.0000,153,156,0.0153,0.0757,18
2,BGL-trained baseline,0.1008,0.1008,153,156,0.0000,0.0010,0
3,OpenStack-trained baseline,0.4727,0.4727,153,156,0.0000,0.0000,0
4,Similar-application baseline,0.3261,0.9229,153,156,0.0000,0.0019,0
5,Isolation Forest alone,0.1626,0.2375,153,156,0.0000,0.0229,11
6,Novelty gate alone,NaN,NaN,153,156,0.0000,0.0019,0
7,Source-free detector on the BGL baseline's fea...,-0.0000,0.0000,153,156,0.0000,0.0010,0


CloudWatch Logs ingestion delay per run and application, in seconds (table ingestion_delay).

In [56]:
rows = []
for r in RUN_TAGS:
    for app in APPS:
        lines = (RUNS / r / app / "raw" / "cloudwatch_events.jsonl").read_text().splitlines()
        d = np.array([(e["ingestionTime"] - e["timestamp"]) / 1000 for e in map(json.loads, lines)])
        rows.append({"run": r, "application": app, "lines": len(d), "median_s": float(np.median(d)),
                     "p95_s": float(np.percentile(d, 95))})
ingestion = save(pd.DataFrame(rows), "ingestion_delay")
ingestion.round(1)

,run,application,lines,median_s,p95_s
0,run1,orders,36059,4.5,8.8
1,run1,bookings,36116,4.7,8.8
2,run2,orders,36154,4.6,8.8
3,run2,bookings,36132,4.4,8.8
4,run3,orders,36128,4.5,8.8
5,run3,bookings,36097,4.6,8.8
6,run4,orders,36019,4.6,8.8
7,run4,bookings,36006,4.4,8.8
8,run5,orders,36104,4.3,8.8
9,run5,bookings,36085,4.4,8.8


Approximate cost of each run at eu-west-1 list prices, derived from the exported logs, the request records and the SageMaker job times (table approximate_cost).

In [57]:
# eu-west-1 list prices in USD (AWS Price List API, read 9 Nov 2026)
PRICE = dict(lambda_request=0.20 / 1e6, gb_second=0.0000166667, write_unit=0.705 / 1e6, read_unit=0.1415 / 1e6,
             log_gb=0.57, sagemaker_hour=0.219)
REPORT = re.compile(r"Billed Duration: (\d+) ms\s+Memory Size: (\d+) MB")
rows = []
for r in RUN_TAGS:
    for app in APPS:
        raw = RUNS / r / app / "raw"
        req = 0
        gbs = 0
        byt = 0
        for line in (raw / "cloudwatch_events.jsonl").read_text().splitlines():
            msg = json.loads(line)["message"]
            byt += len(msg.encode()) + 26
            m = REPORT.search(msg)
            if m:
                req += 1
                gbs += int(m.group(1)) / 1000 * int(m.group(2)) / 1024
        inv = [json.loads(l) for l in (raw / "invocations.jsonl").read_text().splitlines()]
        w = sum(i["method"] == "POST" and i["status_code"] == 201 for i in inv)
        rd = 0.5 * sum(i["method"] == "GET" and i["status_code"] in (200, 404) for i in inv)
        usd = (req * PRICE["lambda_request"] + gbs * PRICE["gb_second"] + w * PRICE["write_unit"]
               + rd * PRICE["read_unit"] + byt / 2**30 * PRICE["log_gb"])
        rows.append({"run": r, "application": app, "lambda_requests": req, "gb_seconds": gbs, "write_units": w,
                     "read_units": rd, "log_mib": byt / 2**20, "usd": usd})
jobs = load(ROOT / "results" / "sagemaker_jobs.json")["jobs"]
seconds = sum((datetime.fromisoformat(j["ended"]) - datetime.fromisoformat(j["created"])).total_seconds() for j in jobs)
rows.append({"run": "SageMaker scoring", "application": f"{len(jobs)} jobs", "usd": seconds / 3600 * PRICE["sagemaker_hour"]})
cost = pd.DataFrame(rows)
cost.loc[len(cost)] = {"run": "total", "usd": cost["usd"].sum()}
save(cost, "approximate_cost")
cost.round(4)

,run,application,lambda_requests,gb_seconds,write_units,read_units,log_mib,usd
0,run1,orders,7106.0,995.8495,3312.0,1657.0,5.0007,0.0234
1,run1,bookings,7120.0,1106.0465,3314.0,1664.0,4.9901,0.0252
2,run2,orders,7127.0,991.9740,3312.0,1667.5,5.0126,0.0233
3,run2,bookings,7122.0,1099.6261,3312.0,1665.0,4.9927,0.0251
4,run3,orders,7122.0,991.9915,3312.0,1665.0,5.0093,0.0233
5,run3,bookings,7114.0,1100.4740,3312.0,1661.0,4.9883,0.0251
6,run4,orders,7099.0,1005.1275,3312.0,1653.5,4.9955,0.0235
7,run4,bookings,7098.0,1108.1368,3314.0,1653.0,4.9759,0.0252
8,run5,orders,7117.0,999.3074,3312.0,1662.5,5.0061,0.0234
9,run5,bookings,7114.0,1104.7335,3312.0,1661.0,4.9863,0.0252
